In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_G.xpt"
csv_file = DATA_DIR / "GHB_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2011-2012 GHB_G XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GHB_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6549:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 2:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,549 rows x 2 columns."
)


# ============================================================
# 4. DEFINE VARIABLE TYPES
# ============================================================

#
# SEQN:
#     identifier / whole-number field
#
# LBXGH:
#     Glycohemoglobin / HbA1c percentage
#     genuine decimal laboratory measurement
#
# IMPORTANT:
#
# DO NOT convert LBXGH to Int64.
# DO NOT remove legitimate ".0".
# DO NOT round LBXGH.
#

INTEGER_COLUMNS = [
    "SEQN"
]


DECIMAL_COLUMNS = [
    "LBXGH"
]


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached GHB_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO numeric zero restoration is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    true_zero_total
)


print(
    "Tiny XPORT values:",
    tiny_total
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT values "
        "found in GHB_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total != 0:

    raise ValueError(
        "STOP: Unexpected numeric zero values "
        "found in GHB_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No unexpected numeric zeros."
)


# ============================================================
# 6. CHECK GENUINE DECIMAL VARIABLES
# ============================================================

#
# This check MUST occur before Int64 conversion.
#
# Expected:
#
#     LBXGH = genuine decimal
#     SEQN  = whole number
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} "
        "fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns != ["LBXGH"]:

    raise ValueError(
        "Unexpected decimal-variable pattern.\n"
        "Expected: ['LBXGH']\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: LBXGH is the only genuine "
    "decimal variable."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. PRESERVE LBXGH EXACTLY
# ============================================================

#
# DO NOT:
#
#     df["LBXGH"].round(...)
#
# DO NOT:
#
#     df["LBXGH"].astype("Int64")
#
# DO NOT:
#
#     use float_format during export
#
# LBXGH must remain its original float values.
#

if not np.allclose(
    source_df["LBXGH"].to_numpy(),
    df["LBXGH"].to_numpy(),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH changed unexpectedly."
    )


print(
    "PASS: LBXGH remains completely unchanged."
)


# ============================================================
# 9. CDC CHECK - LBXGH
# ============================================================

#
# CDC 2011-2012:
#
# LBXGH = Glycohemoglobin (%)
#
# Range:
#
#     3.6 - 17.8
#
# Count:
#
#     6,145
#
# Missing:
#
#     404
#

lbxgh = df["LBXGH"]


print("\n--- LBXGH ---")


print(
    "Minimum:",
    lbxgh.min()
)


print(
    "Maximum:",
    lbxgh.max()
)


print(
    "Non-missing:",
    int(
        lbxgh.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        lbxgh.isna()
        .sum()
    )
)


if not (
    np.isclose(
        lbxgh.min(),
        3.6,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        lbxgh.max(),
        17.8,
        rtol=0,
        atol=0
    )
    and
    int(
        lbxgh.notna()
        .sum()
    ) == 6145
    and
    int(
        lbxgh.isna()
        .sum()
    ) == 404
):

    raise ValueError(
        "LBXGH differs from CDC."
    )


print(
    "PASS: LBXGH matches CDC."
)


# ============================================================
# 10. CHECK LEGITIMATE WHOLE-LOOKING DECIMAL VALUES
# ============================================================

#
# Some HbA1c values are numerically whole numbers:
#
#     4.0
#     5.0
#     6.0
#     7.0
#
# These are STILL genuine decimal laboratory values.
#
# Attached GHB_G contains 559 such observations.
#

whole_looking_lbxgh = np.isclose(
    lbxgh.dropna(),
    np.round(
        lbxgh.dropna()
    ),
    rtol=0,
    atol=1e-12
)


whole_looking_count = int(
    whole_looking_lbxgh.sum()
)


print(
    "\nLBXGH whole-number-looking decimal values:",
    whole_looking_count
)


if whole_looking_count != 559:

    raise ValueError(
        "Unexpected count of whole-looking "
        "LBXGH decimal values."
    )


print(
    "PASS: 559 legitimate values such as "
    "4.0 / 5.0 / 6.0 are preserved."
)


# ============================================================
# 11. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# GHB_G requires NO numeric correction.
#
# The numeric values must therefore remain exactly
# equal to the source XPT.
#
# Only SEQN's CSV representation changes:
#
#     62161.0
#       ->
#     62161
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: Both GHB_G variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 12. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
#
# NO rounding.
#
# This allows LBXGH to retain its natural decimal
# representation.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 13. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if (
    roundtrip_df.shape
    !=
    df.shape
):

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 14. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: Both CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 15. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 16. READ RAW CSV AS TEXT
# ============================================================

#
# Numeric fidelity was checked above.
#
# Now read raw text to check formatting.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 17. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 18. SEQN .0 FORMAT CHECK
# ============================================================

#
# SEQN is the ONLY whole-number variable.
#
# It should never contain:
#
#     62161.0
#
# in the final CSV.
#

seqn_dot_zero = int(
    (
        raw_csv_df["SEQN"].ne("")
        &
        raw_csv_df["SEQN"]
        .str.endswith(".0")
    )
    .sum()
)


print("\n--- SEQN FORMAT CHECK ---")


print(
    "SEQN values ending in .0:",
    seqn_dot_zero
)


if seqn_dot_zero != 0:

    raise ValueError(
        "SEQN still contains unnecessary .0."
    )


print(
    "PASS: SEQN contains no unnecessary .0."
)


# ============================================================
# 19. LBXGH .0 VALUES ARE LEGITIMATE
# ============================================================

#
# DO NOT treat ".0" as an error in LBXGH.
#
# Example:
#
#     6.0
#
# means HbA1c = 6.0%.
#
# It is not an integer questionnaire code.
#

lbxgh_dot_zero = int(
    (
        raw_csv_df["LBXGH"].ne("")
        &
        raw_csv_df["LBXGH"]
        .str.endswith(".0")
    )
    .sum()
)


print("\n--- LBXGH DECIMAL FORMAT CHECK ---")


print(
    "Legitimate LBXGH values ending in .0:",
    lbxgh_dot_zero
)


if lbxgh_dot_zero != 559:

    raise ValueError(
        "Unexpected LBXGH .0 representation count."
    )


print(
    "PASS: Legitimate LBXGH decimal .0 values preserved."
)


# ============================================================
# 20. EXACT LBXGH DECIMAL FIDELITY
# ============================================================

expected_lbxgh = (
    source_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


exported_lbxgh = (
    roundtrip_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_lbxgh,
    exported_lbxgh,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH decimal fidelity failed."
    )


print(
    "PASS: LBXGH decimal values preserved exactly."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "GHB_G CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "LBXGH non-missing:",
    int(
        df["LBXGH"]
        .notna()
        .sum()
    )
)


print(
    "LBXGH missing:",
    int(
        df["LBXGH"]
        .isna()
        .sum()
    )
)


print(
    "LBXGH range:",
    df["LBXGH"].min(),
    "to",
    df["LBXGH"].max()
)


print(
    "Legitimate LBXGH '.0' values:",
    lbxgh_dot_zero
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary SEQN .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 GHB_G XPT -> CSV
Rows: 6,549
Columns: 2
PASS: Structure = 6,549 rows x 2 columns.

--- VARIABLE TYPE PLAN ---
Whole-number variables: ['SEQN']
Genuine decimal variables: ['LBXGH']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT values: 0
PASS: No tiny XPORT artifacts.
PASS: No unexpected numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
LBXGH: 5,586 fractional observations
PASS: LBXGH is the only genuine decimal variable.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.
PASS: LBXGH remains completely unchanged.

--- LBXGH ---
Minimum: 3.6
Maximum: 17.8
Non-missing: 6145
Missing: 404
PASS: LBXGH matches CDC.

LBXGH whole-number-looking decimal values: 559
PASS: 559 legitimate values such as 4.0 / 5.0 / 6.0 are preserved.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: Both GHB_G variables preserve source XPT values exactly.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neigh